# Orbit Wars Physics Sandbox

Companion notebook for `docs/strategy/first-principles.dense.md`.

Reproduces every numerical claim from sections 1–6 with plots.

Engine source of truth: `kaggle_environments/envs/orbit_wars/orbit_wars.py` (cited via `file:line` in the markdown docs).

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

# Constants from orbit_wars.py
BOARD_SIZE = 100.0
CENTER = 50.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
MAX_SPEED = 6.0
COMET_SPEED = 4.0
EPISODE_STEPS = 500
COMET_SPAWN_STEPS = [50, 150, 250, 350, 450]

rng = np.random.default_rng(7240)

## §1. Fleet speed curve

Reproduces the `min(v_max, 1 + (v_max-1)*(ln N / ln 1000)**1.5)` formula at `orbit_wars.py:577-578`.

In [ ]:
def fleet_speed(N, max_speed=MAX_SPEED):
    if N <= 0:
        return 0.0
    s = 1.0 + (max_speed - 1.0) * (math.log(N) / math.log(1000)) ** 1.5
    return min(s, max_speed)


Ns = np.arange(1, 2001)
speeds = np.array([fleet_speed(int(n)) for n in Ns])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Ns, speeds, linewidth=2)
ax[0].axhline(MAX_SPEED, color="red", linestyle="--", alpha=0.5, label=f"cap @ {MAX_SPEED}")
ax[0].axvline(1000, color="gray", linestyle=":", alpha=0.5, label="N=1000 saturation")
ax[0].set_xlabel("Fleet size N")
ax[0].set_ylabel("Speed (units/turn)")
ax[0].set_title("Fleet speed v(N)")
ax[0].legend()
ax[0].grid(alpha=0.3)

# turns to cross 50 units
ax[1].plot(Ns, 50.0 / speeds, linewidth=2, color="C2")
ax[1].axhline(50 / MAX_SPEED, color="red", linestyle="--", alpha=0.5, label="floor @ 8.33 turns")
ax[1].set_xlabel("Fleet size N")
ax[1].set_ylabel("Turns to cross 50 units")
ax[1].set_title("Cross-50u travel time")
ax[1].set_xscale("log")
ax[1].legend()
ax[1].grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("\n  N |  speed | T(50u)")
for n in [1, 5, 10, 50, 100, 250, 500, 1000, 2000]:
    s = fleet_speed(n)
    print(f"{n:5d} | {s:.4f} | {50/s:6.2f}")

## §2. Lead-shot intercept (orbital prediction)

Solve $\|p(t) - a\| = v\,t$ by bisection. Worked example from `first-principles.dense.md` §2.4.

In [ ]:
def lead_intercept(launcher, theta0, omega, r_orb, v):
    """Returns (t_star, fire_angle_rad, intercept_xy)."""

    def pos_at(t):
        a = theta0 + omega * t
        return (CENTER + r_orb * math.cos(a), CENTER + r_orb * math.sin(a))

    def g(t):
        px, py = pos_at(t)
        return math.sqrt((px - launcher[0]) ** 2 + (py - launcher[1]) ** 2) - v * t

    lo, hi = 1e-3, 500.0
    if g(lo) <= 0:
        px, py = pos_at(lo)
        return lo, math.atan2(py - launcher[1], px - launcher[0]), (px, py)
    for _ in range(120):
        mid = 0.5 * (lo + hi)
        if g(mid) > 0:
            lo = mid
        else:
            hi = mid
    t = 0.5 * (lo + hi)
    px, py = pos_at(t)
    return t, math.atan2(py - launcher[1], px - launcher[0]), (px, py)


launcher = (75.0, 25.0)
theta0 = 0.0
omega = 0.04
r_orb = 30.0
ships = 50
v = fleet_speed(ships)

t_star, alpha_star, hit = lead_intercept(launcher, theta0, omega, r_orb, v)
naive_alpha = math.atan2(
    CENTER + r_orb * math.sin(theta0) - launcher[1], CENTER + r_orb * math.cos(theta0) - launcher[0]
)
print(f"fleet speed (50 ships) = {v:.4f}")
print(f"t*               = {t_star:.4f} turns")
print(f"intercept         = ({hit[0]:.4f}, {hit[1]:.4f})")
print(f"fire_angle (lead) = {math.degrees(alpha_star):.4f} deg")
print(f"naive aim         = {math.degrees(naive_alpha):.4f} deg")
print(f"lead correction   = {math.degrees(alpha_star - naive_alpha):.4f} deg")

# Visualize the intercept geometry
fig, ax = plt.subplots(figsize=(7, 7))
# Sun
sun = plt.Circle((CENTER, CENTER), SUN_RADIUS, color="gold", alpha=0.7)
ax.add_patch(sun)
# Orbit
thetas = np.linspace(0, 2 * math.pi, 200)
ax.plot(
    CENTER + r_orb * np.cos(thetas),
    CENTER + r_orb * np.sin(thetas),
    "k--",
    alpha=0.4,
    label="target orbit",
)
# Target trail (during the t* journey)
ts = np.linspace(0, t_star, 60)
tx = CENTER + r_orb * np.cos(theta0 + omega * ts)
ty = CENTER + r_orb * np.sin(theta0 + omega * ts)
ax.plot(tx, ty, "C1-", linewidth=2, label="target path")
ax.plot(tx[0], ty[0], "C1o", markersize=10, label="target @ t=0")
ax.plot(hit[0], hit[1], "C1*", markersize=18, label=f"intercept @ t={t_star:.1f}")
# Fleet path
fx = launcher[0] + v * ts * math.cos(alpha_star)
fy = launcher[1] + v * ts * math.sin(alpha_star)
ax.plot(fx, fy, "C0-", linewidth=2, label="fleet (lead-shot)")
ax.plot(*launcher, "C0s", markersize=10, label="launcher (75,25)")
# Naive fleet path
nfx = launcher[0] + v * ts * math.cos(naive_alpha)
nfy = launcher[1] + v * ts * math.sin(naive_alpha)
ax.plot(nfx, nfy, "r--", linewidth=1, alpha=0.6, label="naive (no lead)")
ax.set_xlim(0, BOARD_SIZE)
ax.set_ylim(0, BOARD_SIZE)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
ax.set_title("Lead-shot vs naive aim (50 ships, omega=0.04, r=30)")
plt.show()

## §2.5 Forbidden cone visualization (sun-cross check)

From `(75, 25)`, the half-angle to the sun is $\arcsin(R_\odot / d) = \arcsin(10/35.36) \approx 16.43^\circ$.

In [ ]:
def forbidden_cone(launcher):
    dx = CENTER - launcher[0]
    dy = CENTER - launcher[1]
    d = math.hypot(dx, dy)
    if d <= SUN_RADIUS:
        return None  # already inside
    theta_sun = math.atan2(dy, dx)
    half = math.asin(SUN_RADIUS / d)
    return theta_sun, half, d


th, half, d_sun = forbidden_cone(launcher)
print(f"launcher→sun distance: {d_sun:.4f}")
print(f"theta_sun: {math.degrees(th):.2f} deg")
print(f"half-cone: {math.degrees(half):.2f} deg")
print(f"forbidden range: [{math.degrees(th-half):.2f}, {math.degrees(th+half):.2f}] deg")

fig, ax = plt.subplots(figsize=(7, 7))
sun = plt.Circle((CENTER, CENTER), SUN_RADIUS, color="gold", alpha=0.7)
ax.add_patch(sun)
ax.plot(*launcher, "C0s", markersize=10, label=f"launcher {launcher}")
# Sample headings to test
alphas = np.linspace(0, 2 * math.pi, 360)
for a in alphas[::8]:
    ex = launcher[0] + 80 * math.cos(a)
    ey = launcher[1] + 80 * math.sin(a)
    # Test point-segment distance to sun
    dx, dy = ex - launcher[0], ey - launcher[1]
    l2 = dx * dx + dy * dy
    t = max(0, min(1, ((CENTER - launcher[0]) * dx + (CENTER - launcher[1]) * dy) / l2))
    px, py = launcher[0] + t * dx, launcher[1] + t * dy
    dist = math.hypot(px - CENTER, py - CENTER)
    color = "red" if dist < SUN_RADIUS else "green"
    ax.plot([launcher[0], ex], [launcher[1], ey], color=color, alpha=0.18, linewidth=0.7)
ax.set_xlim(0, BOARD_SIZE)
ax.set_ylim(0, BOARD_SIZE)
ax.set_aspect("equal")
ax.set_title(f"Forbidden cone from {launcher} (red = sun-kill)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## §3. Comet path geometry (synthetic)

Reproduces `generate_comet_paths` (`orbit_wars.py:191-331`) for one realization. The episode seed is hidden from agents at runtime, but the math itself is deterministic given $(e, a, \phi)$.

In [ ]:
def synthetic_comet_path(e, a, phi, comet_speed=COMET_SPEED, num=5000):
    b = a * math.sqrt(1 - e**2)
    c_val = a * e
    dense = []
    for i in range(num):
        t = 0.3 * math.pi + 1.4 * math.pi * i / (num - 1)
        ex = c_val + a * math.cos(t)
        ey = b * math.sin(t)
        x = CENTER + ex * math.cos(phi) - ey * math.sin(phi)
        y = CENTER + ex * math.sin(phi) + ey * math.cos(phi)
        dense.append((x, y))
    # arc-length resample
    path = [dense[0]]
    cum = 0.0
    target = comet_speed
    for i in range(1, len(dense)):
        d = math.hypot(dense[i][0] - dense[i - 1][0], dense[i][1] - dense[i - 1][1])
        cum += d
        if cum >= target:
            path.append(dense[i])
            target += comet_speed
    # extract on-board segment
    visible = [pt for pt in path if 0 <= pt[0] <= BOARD_SIZE and 0 <= pt[1] <= BOARD_SIZE]
    return visible


fig, ax = plt.subplots(figsize=(8, 8))
sun = plt.Circle((CENTER, CENTER), SUN_RADIUS, color="gold", alpha=0.7)
ax.add_patch(sun)
ax.add_patch(
    plt.Rectangle((0, 0), BOARD_SIZE, BOARD_SIZE, fill=False, edgecolor="black", linewidth=1)
)

for _trial in range(5):
    e = rng.uniform(0.75, 0.93)
    a = rng.uniform(60, 150)
    phi = rng.uniform(math.pi / 6, math.pi / 3)
    visible = synthetic_comet_path(e, a, phi)
    if 5 <= len(visible) <= 40:
        # Plot 4 symmetric copies
        cols = ["C0", "C1", "C2", "C3"]
        # paths follow the engine's symmetry construction (orbit_wars.py:266-269)
        sym_lists = [
            [(y, x) for (x, y) in visible],
            [(BOARD_SIZE - x, y) for (x, y) in visible],
            [(x, BOARD_SIZE - y) for (x, y) in visible],
            [(BOARD_SIZE - y, BOARD_SIZE - x) for (x, y) in visible],
        ]
        for k, sym in enumerate(sym_lists):
            xs = [p[0] for p in sym]
            ys = [p[1] for p in sym]
            ax.plot(xs, ys, color=cols[k], alpha=0.55, linewidth=1.2)
            ax.plot(xs[0], ys[0], "o", color=cols[k], markersize=4)
ax.set_xlim(0, BOARD_SIZE)
ax.set_ylim(0, BOARD_SIZE)
ax.set_aspect("equal")
ax.set_title("5 random comet groups (4 symmetric copies each)")
ax.grid(alpha=0.3)
plt.show()

# Distribution of comet ship counts: min of 4 random U{1..99}
samples = [min(rng.integers(1, 100, 4)) for _ in range(20000)]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(samples, bins=50, color="C3", alpha=0.7, edgecolor="black", linewidth=0.4)
ax.axvline(
    np.median(samples), color="black", linestyle="--", label=f"median={np.median(samples):.0f}"
)
ax.axvline(
    np.percentile(samples, 90),
    color="red",
    linestyle=":",
    label=f"P90={np.percentile(samples,90):.0f}",
)
ax.set_xlabel("Comet defending ships")
ax.set_ylabel("count")
ax.set_title("Comet ship distribution = min of 4 uniform[1,99]")
ax.legend()
plt.show()

## §4. Combat surplus heatmap

Outcome of $G$ vs $T$ vs $S$ where $T \ge S$ (top vs second attacker, vs garrison).

In [ ]:
def combat_outcome(G, T, S):
    top, sec = (T, S) if T >= S else (S, T)
    if top == sec:
        return 0  # tie, garrison untouched
    survivor = top - sec
    if survivor <= G:
        return -survivor  # garrison reduced
    return survivor - G  # planet flipped, new garrison


Ts = np.arange(0, 201, 4)
Ss = np.arange(0, 201, 4)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, G in zip(axes, [20, 50, 80], strict=False):
    grid = np.zeros((len(Ss), len(Ts)))
    for i, S in enumerate(Ss):
        for j, T in enumerate(Ts):
            grid[i, j] = combat_outcome(G, T, S)
    vmax = max(abs(grid.min()), grid.max())
    im = ax.imshow(
        grid,
        origin="lower",
        extent=[0, 200, 0, 200],
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        aspect="auto",
    )
    ax.set_xlabel("Top attacker T")
    ax.set_ylabel("Second attacker S")
    ax.set_title(f"G={G} (red=flip+surplus, blue=garrison loss)")
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Tie diagonal: T = S => survivor = 0
# Flip line: T - S = G + 1
print("Cheapest flip when S=0:  T = G + 1")
print("Cheapest flip when S>0:  T = G + S + 1")

## §5. Home-to-home distance distribution

Sample the engine's home-placement chain (Phase 1 static class, `orbit_wars.py:67-122`).

In [ ]:
def sample_home_distance(rng):
    while True:
        prod = rng.integers(1, 6)
        r = 1 + math.log(int(prod))
        angle = rng.uniform(0, math.pi / 2)
        min_orb = ROTATION_RADIUS_LIMIT - r
        max_orb = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orb > max_orb:
            continue
        orb = rng.uniform(min_orb, max_orb)
        x_src = CENTER + orb * math.cos(angle)
        y_src = CENTER + orb * math.sin(angle)
        if (x_src - CENTER) < r + 5 or (y_src - CENTER) < r + 5:
            continue
        # storage swap (orbit_wars.py:103): planet[2]=y_src, planet[3]=x_src
        p2, p3 = y_src, x_src
        q4_2, q4_3 = BOARD_SIZE - p2, BOARD_SIZE - p3
        return math.hypot(p2 - q4_2, p3 - q4_3)


Ds = [sample_home_distance(rng) for _ in range(50000)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(Ds, bins=80, color="C2", edgecolor="black", linewidth=0.3, alpha=0.75)
ax.axvline(np.median(Ds), color="black", linestyle="--", label=f"median={np.median(Ds):.2f}")
ax.axvline(np.mean(Ds), color="red", linestyle=":", label=f"mean={np.mean(Ds):.2f}")
ax.set_xlabel("Home-to-home distance (units)")
ax.set_ylabel("count")
ax.set_title(f"Home-to-home distance (n={len(Ds)})")
ax.legend()
plt.show()
print(f"min={min(Ds):.2f}  median={np.median(Ds):.2f}  mean={np.mean(Ds):.2f}  max={max(Ds):.2f}")
print(
    f"Time to cross at v(1000)=6: {np.median(Ds)/MAX_SPEED:.2f} turns ({100*np.median(Ds)/MAX_SPEED/EPISODE_STEPS:.1f}% of episode)"
)

## §6. Production ROI vs distance (break-even surface)

$T_{\text{breakeven}}(G, \text{prod}, D) = D/v(G+1) + (G+1)/\text{prod}$. Lower = more cost-effective.

In [ ]:
garrisons = np.arange(5, 100, 2)
prods = [1, 2, 3, 4, 5]
distances = [25, 50, 75, 100]

fig, axes = plt.subplots(1, len(distances), figsize=(16, 4), sharey=True)
for ax, D in zip(axes, distances, strict=False):
    for prod in prods:
        breakeven = []
        for G in garrisons:
            S = G + 1
            travel = D / fleet_speed(int(S))
            payback = S / prod
            breakeven.append(travel + payback)
        ax.plot(garrisons, breakeven, label=f"prod={prod}", linewidth=2)
    ax.set_xlabel("Garrison G")
    ax.set_title(f"Distance D={D}")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Total break-even time (turns)")
plt.suptitle("Capture cost + payback vs garrison, by production tier")
plt.tight_layout()
plt.show()

## §7. Toy opener: 1-step beam over pruned actions

Quick demonstration that a small action set (8 pruned moves) is searchable, vs the unpruned $\sim 10^{19}$ raw.

In [ ]:
# Simulate a tiny opening state: home at (75,25) with 10 ships, 4 nearby targets.
from dataclasses import dataclass


@dataclass
class P:
    x: float
    y: float
    ships: int
    prod: int
    owner: int


home = P(75.0, 25.0, 10, 3, 0)
neutral = [
    P(60.0, 30.0, 12, 2, -1),
    P(80.0, 50.0, 25, 3, -1),
    P(90.0, 25.0, 8, 1, -1),
    P(50.0, 30.0, 30, 5, -1),
]


# Score function: ships you'd own after capturing a target with given fleet
def score_capture(home, target, fleet):
    if fleet >= target.ships + 1:
        # Approximate flight time using fleet_speed
        d = math.hypot(home.x - target.x, home.y - target.y)
        t = d / fleet_speed(fleet)
        # Captured planet contributes (turns_remaining_in_first_50 * prod) ships
        residual_turns = max(0, 50 - t)
        captured = (fleet - target.ships) + residual_turns * target.prod
        # Plus residual home production
        return captured + 50 * home.prod - fleet
    return -1e9  # failed flip


# Try every (target, ships_sent) pair; ships_sent in {target.ships+1, all_ships}
candidates = []
for tgt in neutral:
    for ships in [tgt.ships + 1, home.ships]:
        if ships <= home.ships:
            s = score_capture(home, tgt, ships)
            candidates.append((s, tgt, ships))

candidates.sort(reverse=True, key=lambda x: x[0])
print("Top 4 first moves (score | target | ships):")
for s, t, ships in candidates[:4]:
    d = math.hypot(home.x - t.x, home.y - t.y)
    arrive = d / fleet_speed(ships) if ships > 0 else float("inf")
    print(
        f"  score={s:7.2f}  tgt=({t.x},{t.y}) prod={t.prod} G={t.ships}  send={ships}  arrive={arrive:.2f}t"
    )

# Visualize
fig, ax = plt.subplots(figsize=(7, 7))
sun = plt.Circle((CENTER, CENTER), SUN_RADIUS, color="gold", alpha=0.7)
ax.add_patch(sun)
ax.plot(home.x, home.y, "C0s", markersize=14, label="home (10 ships, prod=3)")
for t in neutral:
    ax.plot(t.x, t.y, "ko", markersize=6 + t.prod * 2, alpha=0.7)
    ax.annotate(
        f"prod={t.prod}\nG={t.ships}",
        (t.x, t.y),
        textcoords="offset points",
        xytext=(8, 8),
        fontsize=8,
    )
best_s, best_t, best_ships = candidates[0]
ax.annotate(
    "",
    xy=(best_t.x, best_t.y),
    xytext=(home.x, home.y),
    arrowprops=dict(arrowstyle="->", color="C0", lw=2),
)
ax.set_xlim(40, 100)
ax.set_ylim(0, 60)
ax.set_aspect("equal")
ax.set_title(f"Greedy opener: send {best_ships} to ({best_t.x},{best_t.y})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Summary

Every cell above derives a numerical fact also reported in `docs/strategy/first-principles.dense.md`. Cross-checks:

- §1 fleet speed table matches §1.2 of the dense doc.
- §2 lead-shot example matches §2.4 numerics ($t^* \approx 12.63$, fire angle $\approx 88.18^\circ$).
- §2.5 forbidden cone half-angle $\approx 16.43^\circ$ from $(75, 25)$.
- §3 comet ship distribution: median $\approx 19$, P90 $\approx 53$.
- §5 home-to-home median $\approx 100$.
- §6 prod=3+ planets dominate ROI even at distance 100.